In [4]:
import fitz
from langchain_core.documents import Document
from transformers import CLIPModel, CLIPProcessor, CLIPTextModel
from PIL import Image
import torch
import numpy as np
from langchain_classic.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.schema.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

c:\Users\Sakshi Sinha\Downloads\RAG\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import transformers
print(transformers.__version__)

4.41.2


In [6]:
#Loading the clip model
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [8]:
#Instantialte the clip model
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [9]:
print(type(clip_model))
print(clip_model)

# features = clip_model.get_text_features(**inputs)
# print(type(features))

<class 'transformers.models.clip.modeling_clip.CLIPModel'>
CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
      

In [10]:
#Embedding functions
def embedding_image(image_data):
    """EMbed using CLIP"""
    if isinstance(image_data, str): #If path
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data
    
    inputs = clip_processor(images = image, return_tensors = "pt" )
    with torch.no_grad():

        features = clip_model.get_image_features(**inputs)
        #Normalize the embeddings to unit vector
        features = features /features.norm(dim= -1, keepdim = True)
        return features.squeeze().numpy()

def embed_text(text):
    """Embed the text using CLIP"""
    inputs = clip_processor(
        text = text,
        return_tensors = "pt",
        padding = True,
        truncation = True,
        max_length = 77   #Clip max token length
    )
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        print(type(features))
        # Normalize embeddings
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()


In [11]:
#Process PDF
pdf_path = "multimodal_sample.pdf"
doc = fitz.open(pdf_path)

#Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {} #Store actual image data for llm

#Text splitte 
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)


In [12]:
doc 

Document('multimodal_sample.pdf')

In [13]:
from langchain_core.documents import Document

In [14]:
for i, page in enumerate(doc):
    #Process the text
    text = page.get_text()
    if text.strip():
        #create temp document for splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        #EMbed the chunks using clip
        for chunk in text_chunks:
            embeddings = embed_text(chunk.page_content)
            all_embeddings.append(embeddings)
            all_docs.append(chunk)

    
    ## process images
    ##Three Important Actions:

    ##Convert PDF image to PIL format
    ##Store as base64 for GPT-4V (which needs base64 images)
    ##Create CLIP embedding for retrieval

    for img_indx, image in enumerate(page.get_images(full = True)):
        try:
            xref = image[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            #Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            #create unique identifier
            image_id = f"page_{i}_img_{img_indx}"

            #Store image as base 64 for later use
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")

            image_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = image_base64

            #Embbed image using clip
            embedding = embedding_image(pil_image)
            all_embeddings.append(embedding)

            #create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)
        
        except Exception as e:
            print(f"Error processing image {img_indx} on page {i}: {e}")
            continue


doc.close()
            




<class 'torch.Tensor'>


In [15]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [16]:
(all_docs, all_embeddings)

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
  Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')],
 [array([-2.67243758e-03,  1.28300162e-02, -5.18314131e-02,  4.14879657e-02,
         -2.33941823e-02, -7.55864382e-03, -3.67659368e-02,  1.19710736e-01,
          8.52080807e-02,  2.05424847e-03, -1.11534819e-02, -1.29592428e-02,
          5.25014587e-02, -3.65392631e-03,  4.76078913e-02,  1.58372745e-02,
          2.03388184e-02,  4.35362123e-02, -3.29169258e-03,  2.03181859e-02,
          1.88024947e-03, -4.23493683e-02,  5.44102723e-03,  3

In [17]:
embedding_array = np.array(all_embeddings)
embedding_array

array([[-0.00267244,  0.01283002, -0.05183141, ..., -0.0038509 ,
         0.02977717, -0.00010682],
       [ 0.01732339, -0.01327691, -0.02427033, ...,  0.0899405 ,
        -0.00272155,  0.03253037]], shape=(2, 512), dtype=float32)

In [18]:
#vectore store from embeddings
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embedding_array)],
    embedding=None,
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [19]:
llm = init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018DD3106230>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018DD32517B0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [20]:
def retrive_multilomdel(query, k=5):
    """Unified retrieval using clip embeddings"""
    # Embed query using CLIP
    query_embedding = embed_text(query)
    
    # Search in unified vector store
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    
    return results

In [21]:
def create_multimodel_message(query, retrieved_docs):
    """Create a message with both text and images for GPT-4V."""
    content = []

    #Add the query
    content.append({
        "type" : "text",
        "text": f"Question : {query} \n\n content: \n"
    })

    #Seperate text and image document
    text_docs = [docs for docs in retrieved_docs if docs.metadata.get("type")=="text"]
    image_docs = [docs for docs in retrieved_docs if docs.metadata.get("type"=="image")]

    #Add text context
    if text_docs:
        text_context = "\n\n".join(
            [
                f"[Page {doc.metadata['page']}] : {doc.page_content}" for doc in text_docs
            ]
        )
        content.append({
            "type" : "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    
    #Add image document
    if image_docs:
        for docs in image_docs:
            image_id = docs.metadata.get("image_id")
            if image_id and image_id in image_data_store:
                content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })
    
    return HumanMessage(content=content)

In [22]:
def multimodel_rag_pipeline(query):
    """Main pipeline for multimodel rag"""

    #Retrive relevant documents
    retrive_docs = retrive_multilomdel(query)

    #create multimodel message:
    multimodel_message = create_multimodel_message(query, retrive_docs)
    # Get response from GPT-4V
    response = llm.invoke([multimodel_message])
    # Print retrieved context info
    print(f"\nRetrieved {len(retrive_docs)} documents:")
    for doc in retrive_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    return response.content

In [24]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodel_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------
<class 'torch.Tensor'>

Retrieved 2 documents:
  - Text from page 0: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
  - Image from page 0


Answer: Based on the provided text, the chart on page 1 appears to show the following revenue trends:

- Q1 (First Quarter): A moderate increase in revenue.
- Q2 (Second Quarter): Revenue growth outperformed Q1 due to marketing campaigns.
- Q3 (Third Quarter): Exponential growth due to global expansion (the highest growth recorded).

The chart likely illustrates a line or bar graph showing the revenue for each quarter, with Q3 having the steepest or most significant increase.

Query: Summarize the main findings from the document
--------------------------------------------------
<class 'torch.Tensor'>

Retrieved 2 documents:
  - Text from page 0: Annual Revenue Overview
This doc